In [3]:

import pandas as pd
from pathlib import Path

# Load the saved invoice-level data
file_path = Path("../data/processed/invoice_level.csv")

invoices = pd.read_csv(
    file_path,
    parse_dates=["InvoiceDate"]
)

print("Dataset shape:", invoices.shape)
print("Unique customers:", invoices["Customer ID"].nunique())

invoices.head()


Dataset shape: (36969, 6)
Unique customers: 5878


,Customer ID,Invoice,InvoiceDate,InvoiceValue,ItemsPurchased,ProductTypes
0,12346.0,491725,2009-12-14 08:34:00,45.0,10,1
1,12346.0,491742,2009-12-14 11:00:00,22.5,5,1
2,12346.0,491744,2009-12-14 11:02:00,22.5,5,1
3,12346.0,492718,2009-12-18 10:47:00,22.5,5,1
4,12346.0,492722,2009-12-18 10:55:00,1.0,1,1


In [4]:

import pandas as pd

snapshot_date = pd.Timestamp("2011-06-01")
prediction_end = snapshot_date + pd.Timedelta(days=90)

# Historical invoices: information available before the snapshot
history = invoices[invoices["InvoiceDate"] < snapshot_date].copy()

# Future invoices: used only to create the target label
future = invoices[
    (invoices["InvoiceDate"] >= snapshot_date) &
    (invoices["InvoiceDate"] < prediction_end)
].copy()

print("Snapshot date:", snapshot_date.date())
print("Prediction window ends:", prediction_end.date())
print("Historical invoices:", len(history))
print("Future invoices for labeling:", len(future))


Snapshot date: 2011-06-01
Prediction window ends: 2011-08-30
Historical invoices: 25846
Future invoices for labeling: 3939


In [5]:

# Define historical windows
history_90d = history[
    history["InvoiceDate"] >= snapshot_date - pd.Timedelta(days=90)
]

history_180d = history[
    history["InvoiceDate"] >= snapshot_date - pd.Timedelta(days=180)
]

# Features calculated from all historical invoices
features = (
    history.groupby("Customer ID")
    .agg(
        last_purchase=("InvoiceDate", "max"),
        total_invoices=("Invoice", "nunique")
    )
    .reset_index()
)

# Recency in days
features["recency_days"] = (
    snapshot_date - features["last_purchase"]
).dt.days

# Recent purchase frequency
frequency_90d = (
    history_90d.groupby("Customer ID")["Invoice"].nunique()
    .rename("frequency_90d")
)

frequency_180d = (
    history_180d.groupby("Customer ID")["Invoice"].nunique()
    .rename("frequency_180d")
)

monetary_180d = (
    history_180d.groupby("Customer ID")["InvoiceValue"].sum()
    .rename("monetary_180d")
)

features = features.join(frequency_90d, on="Customer ID")
features = features.join(frequency_180d, on="Customer ID")
features = features.join(monetary_180d, on="Customer ID")

# Customers with no invoices in a window get zero
features[["frequency_90d", "frequency_180d", "monetary_180d"]] = (
    features[["frequency_90d", "frequency_180d", "monetary_180d"]]
    .fillna(0)
)

features.head()


,Customer ID,last_purchase,total_invoices,recency_days,frequency_90d,frequency_180d,monetary_180d
0,12346.0,2011-01-18 10:01:00,12,133,0.0,1.0,77183.60
1,12347.0,2011-04-07 10:43:00,4,54,1.0,3.0,2535.22
2,12348.0,2011-04-05 10:47:00,4,56,1.0,3.0,1487.24
3,12349.0,2010-10-28 08:23:00,3,215,0.0,0.0,0.00
4,12350.0,2011-02-02 16:01:00,1,118,0.0,1.0,334.40


In [6]:

# Identify customers with a qualifying purchase in the next 90 days
future_customers = future["Customer ID"].unique()

features["buys_again_90d"] = (
    features["Customer ID"].isin(future_customers).astype(int)
)

print("Customer feature rows:", len(features))
print("\nTarget distribution:")
print(features["buys_again_90d"].value_counts())
print("\nTarget proportions:")
print(features["buys_again_90d"].value_counts(normalize=True).round(3))

features.head()


Customer feature rows: 4933

Target distribution:
buys_again_90d
0    3312
1    1621
Name: count, dtype: int64

Target proportions:
buys_again_90d
0    0.671
1    0.329
Name: proportion, dtype: float64


,Customer ID,last_purchase,total_invoices,recency_days,frequency_90d,frequency_180d,monetary_180d,buys_again_90d
0,12346.0,2011-01-18 10:01:00,12,133,0.0,1.0,77183.60,0
1,12347.0,2011-04-07 10:43:00,4,54,1.0,3.0,2535.22,1
2,12348.0,2011-04-05 10:47:00,4,56,1.0,3.0,1487.24,0
3,12349.0,2010-10-28 08:23:00,3,215,0.0,0.0,0.00,0
4,12350.0,2011-02-02 16:01:00,1,118,0.0,1.0,334.40,0


In [7]:

snapshot_date = pd.Timestamp("2011-06-01")
prediction_end = snapshot_date + pd.Timedelta(days=90)

history = invoices[
    invoices["InvoiceDate"] < snapshot_date
].copy()

future = invoices[
    (invoices["InvoiceDate"] >= snapshot_date) &
    (invoices["InvoiceDate"] < prediction_end)
].copy()

print("Snapshot date:", snapshot_date.date())
print("Prediction window ends (exclusive):", prediction_end.date())
print("Historical invoices:", len(history))
print("Future invoices:", len(future))


Snapshot date: 2011-06-01
Prediction window ends (exclusive): 2011-08-30
Historical invoices: 25846
Future invoices: 3939


In [8]:

import pandas as pd

invoices["InvoiceDate"] = pd.to_datetime(invoices["InvoiceDate"])

snapshot_dates = [
    "2010-09-01",
    "2010-12-01",
    "2011-03-01",
    "2011-06-01",
    "2011-09-01",
]

def create_snapshot(invoices, date):
    snapshot_date = pd.Timestamp(date)
    prediction_end = snapshot_date + pd.Timedelta(days=90)

    # Use only information available before the snapshot
    history = invoices[invoices["InvoiceDate"] < snapshot_date]

    # Future purchases are used only to create the label
    future = invoices[
        (invoices["InvoiceDate"] >= snapshot_date) &
        (invoices["InvoiceDate"] < prediction_end)
    ]

    # Customer features from historical invoices
    features = (
        history.groupby("Customer ID")
        .agg(
            last_purchase=("InvoiceDate", "max"),
            total_invoices=("Invoice", "nunique"),
        )
        .reset_index()
    )

    features["recency_days"] = (
        snapshot_date - features["last_purchase"]
    ).dt.days

    for days in [90, 180]:
        recent = history[
            history["InvoiceDate"] >=
            snapshot_date - pd.Timedelta(days=days)
        ]

        counts = recent.groupby("Customer ID")["Invoice"].nunique()

        features[f"frequency_{days}d"] = (
            features["Customer ID"].map(counts).fillna(0)
        )

    recent_180 = history[
        history["InvoiceDate"] >=
        snapshot_date - pd.Timedelta(days=180)
    ]

    spending = recent_180.groupby("Customer ID")["InvoiceValue"].sum()

    features["monetary_180d"] = (
        features["Customer ID"].map(spending).fillna(0)
    )

    # Label: at least one purchase during the next 90 days
    future_customers = set(future["Customer ID"].unique())

    features["buys_again_90d"] = (
        features["Customer ID"].isin(future_customers).astype(int)
    )

    features["snapshot_date"] = snapshot_date

    return features


snapshots = pd.concat(
    [create_snapshot(invoices, date) for date in snapshot_dates],
    ignore_index=True
)

print("Snapshot dataset shape:", snapshots.shape)
print("\nRows per snapshot:")
print(snapshots.groupby("snapshot_date").size())

print("\nTarget distribution:")
print(snapshots["buys_again_90d"].value_counts())


Snapshot dataset shape: (22306, 9)

Rows per snapshot:
snapshot_date
2010-09-01    3321
2010-12-01    4266
2011-03-01    4537
2011-06-01    4933
2011-09-01    5249
dtype: int64

Target distribution:
buys_again_90d
0    13517
1     8789
Name: count, dtype: int64


In [9]:

output_path = Path("../data/processed/customer_snapshots.csv")
snapshots.to_csv(output_path, index=False)

print("Saved:", output_path.resolve())


Saved: C:\Users\vikes\Desktop\Major project\data\processed\customer_snapshots.csv


In [10]:

print(
    snapshots.groupby("snapshot_date")["buys_again_90d"]
    .agg(["count", "sum", "mean"])
    .round(3)
)


               count   sum   mean
snapshot_date                    
2010-09-01      3321  1929  0.581
2010-12-01      4266  1411  0.331
2011-03-01      4537  1587  0.350
2011-06-01      4933  1621  0.329
2011-09-01      5249  2241  0.427


In [11]:

feature_columns = [
    "recency_days",
    "total_invoices",
    "frequency_90d",
    "frequency_180d",
    "monetary_180d",
]

train = snapshots[
    snapshots["snapshot_date"].isin(
        [pd.Timestamp("2010-09-01"), pd.Timestamp("2010-12-01")]
    )
].copy()

test = snapshots[
    snapshots["snapshot_date"] == pd.Timestamp("2011-09-01")
].copy()

X_train = train[feature_columns]
y_train = train["buys_again_90d"]

X_test = test[feature_columns]
y_test = test["buys_again_90d"]

print("Training rows:", len(train))
print("Test rows:", len(test))
print("Training positive rate:", round(y_train.mean(), 3))
print("Test positive rate:", round(y_test.mean(), 3))


Training rows: 7587
Test rows: 5249
Training positive rate: 0.44
Test positive rate: 0.427


In [12]:

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    average_precision_score,
    confusion_matrix,
)

model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced")
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]

print("Classification report:")
print(classification_report(y_test, predictions, zero_division=0))

print("Confusion matrix:")
print(confusion_matrix(y_test, predictions))

print("Average precision (PR-AUC summary):",
      round(average_precision_score(y_test, probabilities), 3))


Classification report:
              precision    recall  f1-score   support

           0       0.72      0.87      0.78      3008
           1       0.75      0.54      0.63      2241

    accuracy                           0.73      5249
   macro avg       0.73      0.70      0.71      5249
weighted avg       0.73      0.73      0.72      5249

Confusion matrix:
[[2602  406]
 [1033 1208]]
Average precision (PR-AUC summary): 0.751


In [13]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
)

rf_model = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
)

rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)
rf_probabilities = rf_model.predict_proba(X_test)[:, 1]

rf_results = {
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_test, rf_predictions),
    "Precision": precision_score(y_test, rf_predictions, zero_division=0),
    "Recall": recall_score(y_test, rf_predictions, zero_division=0),
    "F1": f1_score(y_test, rf_predictions, zero_division=0),
    "Average Precision": average_precision_score(y_test, rf_probabilities),
}

print(pd.Series(rf_results).to_string())


Model                Random Forest
Accuracy                  0.724138
Precision                 0.733373
Recall                    0.556002
F1                        0.632487
Average Precision         0.742316


In [14]:

from sklearn.metrics import precision_score, recall_score, f1_score

for threshold in [0.30, 0.40, 0.50, 0.60, 0.70]:
    threshold_predictions = (
        probabilities >= threshold
    ).astype(int)

    print(
        f"Threshold: {threshold:.2f} | "
        f"Precision: {precision_score(y_test, threshold_predictions):.3f} | "
        f"Recall: {recall_score(y_test, threshold_predictions):.3f} | "
        f"F1: {f1_score(y_test, threshold_predictions):.3f}"
    )


Threshold: 0.30 | Precision: 0.581 | Recall: 0.821 | F1: 0.680
Threshold: 0.40 | Precision: 0.656 | Recall: 0.711 | F1: 0.682
Threshold: 0.50 | Precision: 0.748 | Recall: 0.539 | F1: 0.627
Threshold: 0.60 | Precision: 0.812 | Recall: 0.403 | F1: 0.539
Threshold: 0.70 | Precision: 0.862 | Recall: 0.302 | F1: 0.447


In [15]:

# Make sure snapshot dates are in datetime format
snapshots["snapshot_date"] = pd.to_datetime(snapshots["snapshot_date"])

# Training data: earlier dates
train = snapshots[
    snapshots["snapshot_date"].isin([
        pd.Timestamp("2010-09-01"),
        pd.Timestamp("2010-12-01"),
        pd.Timestamp("2011-03-01")
    ])
]

# Validation data: choose the best threshold
validation = snapshots[
    snapshots["snapshot_date"] == pd.Timestamp("2011-06-01")
]

# Test data: final evaluation
test = snapshots[
    snapshots["snapshot_date"] == pd.Timestamp("2011-09-01")
]

# Select input features and target
X_train = train[feature_columns]
y_train = train["buys_again_90d"]

X_val = validation[feature_columns]
y_val = validation["buys_again_90d"]

X_test = test[feature_columns]
y_test = test["buys_again_90d"]

print("Training rows:", len(train))
print("Validation rows:", len(validation))
print("Testing rows:", len(test))


Training rows: 12124
Validation rows: 4933
Testing rows: 5249


In [16]:

from sklearn.metrics import precision_score, recall_score, f1_score

# Train Logistic Regression
model.fit(X_train, y_train)

# Get probabilities for validation customers
val_probabilities = model.predict_proba(X_val)[:, 1]

# Compare different thresholds
for threshold in [0.30, 0.40, 0.50, 0.60, 0.70]:
    val_predictions = (val_probabilities >= threshold).astype(int)

    print(
        "Threshold:", threshold,
        "| Precision:", round(precision_score(y_val, val_predictions, zero_division=0), 3),
        "| Recall:", round(recall_score(y_val, val_predictions, zero_division=0), 3),
        "| F1:", round(f1_score(y_val, val_predictions, zero_division=0), 3)
    )


Threshold: 0.3 | Precision: 0.467 | Recall: 0.878 | F1: 0.61
Threshold: 0.4 | Precision: 0.54 | Recall: 0.78 | F1: 0.639
Threshold: 0.5 | Precision: 0.647 | Recall: 0.645 | F1: 0.646
Threshold: 0.6 | Precision: 0.74 | Recall: 0.493 | F1: 0.592
Threshold: 0.7 | Precision: 0.813 | Recall: 0.367 | F1: 0.506


In [17]:

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    average_precision_score
)

# Predict probabilities for test customers
test_probabilities = model.predict_proba(X_test)[:, 1]

# Use the threshold selected from validation
threshold = 0.50
test_predictions = (test_probabilities >= threshold).astype(int)

# Show model performance
print("Accuracy:", round(accuracy_score(y_test, test_predictions), 3))
print("\nClassification Report:")
print(classification_report(y_test, test_predictions))

print("Confusion Matrix:")
print(confusion_matrix(y_test, test_predictions))

print(
    "Average Precision:",
    round(average_precision_score(y_test, test_probabilities), 3)
)


Accuracy: 0.724

Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.85      0.78      3008
           1       0.73      0.56      0.63      2241

    accuracy                           0.72      5249
   macro avg       0.73      0.70      0.71      5249
weighted avg       0.73      0.72      0.72      5249

Confusion Matrix:
[[2548  460]
 [ 987 1254]]
Average Precision: 0.751


In [18]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, precision_score, recall_score

rf_model = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
)

rf_model.fit(X_train, y_train)

# Choose threshold using validation data
val_probs = rf_model.predict_proba(X_val)[:, 1]
val_preds = (val_probs >= 0.50).astype(int)

print("Validation F1:", round(f1_score(y_val, val_preds), 3))
print("Validation Precision:", round(precision_score(y_val, val_preds), 3))
print("Validation Recall:", round(recall_score(y_val, val_preds), 3))


Validation F1: 0.637
Validation Precision: 0.649
Validation Recall: 0.625


In [19]:

from sklearn.metrics import (
    accuracy_score, f1_score,
    precision_score, recall_score,
    average_precision_score
)

# Random Forest predictions
rf_test_probs = rf_model.predict_proba(X_test)[:, 1]
rf_test_preds = (rf_test_probs >= 0.50).astype(int)

print("Random Forest Test Results")
print("Accuracy:", round(accuracy_score(y_test, rf_test_preds), 3))
print("Precision:", round(precision_score(y_test, rf_test_preds), 3))
print("Recall:", round(recall_score(y_test, rf_test_preds), 3))
print("F1:", round(f1_score(y_test, rf_test_preds), 3))
print("Average Precision:", round(
    average_precision_score(y_test, rf_test_probs), 3
))


Random Forest Test Results
Accuracy: 0.725
Precision: 0.743
Recall: 0.544
F1: 0.629
Average Precision: 0.751


In [20]:

import pandas as pd

rf = rf_model.named_steps["randomforestclassifier"]

importance = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(
    "Importance", ascending=False
)

print(importance)


          Feature  Importance
4   monetary_180d    0.311154
0    recency_days    0.228242
3  frequency_180d    0.200540
1  total_invoices    0.190435
2   frequency_90d    0.069629


In [21]:

import joblib
from pathlib import Path

# Create the models folder if it doesn't exist
model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)

# Save Logistic Regression
joblib.dump(
    model,
    model_dir / "customer_purchase_model.pkl"
)

# Save feature names
joblib.dump(
    feature_columns,
    model_dir / "feature_columns.pkl"
)

print("Model and feature names saved successfully.")


Model and feature names saved successfully.
